# RBI regulations Local RAG Pipeline

This notebook implements a step-by-step local Retrieval-Augmented Generation (RAG) pipeline from scratch, adapted from [Daniel Bourke's simple-local-rag](https://github.com/mrdbourke/simple-local-rag).

Instead of human nutrition, we are index and query the **Reserve Bank of India (RBI) NBFC Master Directions** PDF (`RBIdoc.pdf`) to build a regulatory compliance chatbot for fintech environments.

## Pipeline Architecture:
1. **PDF Ingestion & Processing**: Extract text from RBI PDF page-by-page using PyMuPDF (`fitz`).
2. **Sentence Segmentation**: Split text into sentences using `spaCy`'s sentencizer.
3. **Text Chunking**: Group sentences into paragraph-sized chunks (e.g. 10 sentences each).
4. **Vector Embedding**: Translate text chunks into numerical vectors using Hugging Face sentence-transformers (CPU-optimized).
5. **Similarity Search**: Implement vector search using dot product / cosine similarity via PyTorch.
6. **Augmentation**: Construct a contextual prompt containing the query and retrieved document passages.
7. **Generation**: Generate an answer using a local LLM or the Gemini API as a fast CPU fallback.

### 1. Requirements Setup

In [ ]:
# Installs (if needed)
# !pip install PyMuPDF sentence-transformers pandas numpy tqdm transformers accelerate google-generativeai spacy

### 2. PDF Ingestion & Text Extraction

In [ ]:
import os
import fitz
from tqdm.auto import tqdm

pdf_path = "RBIdoc.pdf"

if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"{pdf_path} not found in current directory. Please make sure the RBI regulations PDF is copied here.")
else:
    print(f"Found PDF file: {pdf_path}")

In [ ]:
def text_formatter(text: str) -> str:
    """Performs minor formatting on text."""
    cleaned_text = text.replace("\n", " ").strip()
    return cleaned_text

def open_and_read_pdf(pdf_path: str) -> list[dict]:
    """
    Opens a PDF file, reads its text content page by page, and collects statistics.
    """
    doc = fitz.open(pdf_path)
    pages_and_texts = []
    for page_number, page in tqdm(enumerate(doc)):
        text = page.get_text()
        text = text_formatter(text)
        pages_and_texts.append({
            "page_number": page_number, 
            "page_char_count": len(text),
            "page_word_count": len(text.split(" ")),
            "page_sentence_count_raw": len(text.split(". ")),
            "page_token_count": len(text) / 4,
            "text": text
        })
    return pages_and_texts

pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)
print(f"Loaded {len(pages_and_texts)} pages.")

In [ ]:
import random
random.sample(pages_and_texts, k=2)

In [ ]:
import pandas as pd
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

### 3. Sentence Segmentation via spaCy

In [ ]:
from spacy.lang.en import English

nlp = English()
nlp.add_pipe("sentencizer")

# Segment each page's text into sentences
for item in tqdm(pages_and_texts):
    item["sentences"] = list(nlp(item["text"]).sents)
    # Convert Spacy span objects to raw strings
    item["sentences"] = [str(sentence) for sentence in item["sentences"]]
    item["page_sentence_count_spacy"] = len(item["sentences"])

df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

### 4. Grouping Sentences into Chunks

In [ ]:
num_sentence_chunk_size = 10

def split_list(input_list: list, slice_size: int) -> list[list[str]]:
    return [input_list[i:i + slice_size] for i in range(0, len(input_list), slice_size)]

for item in tqdm(pages_and_texts):
    item["sentence_chunks"] = split_list(input_list=item["sentences"], slice_size=num_sentence_chunk_size)
    item["num_chunks"] = len(item["sentence_chunks"])

In [ ]:
import re

pages_and_chunks = []
for item in tqdm(pages_and_texts):
    for sentence_chunk in item["sentence_chunks"]:
        chunk_dict = {}
        chunk_dict["page_number"] = item["page_number"]
        
        # Format chunk string
        joined_sentence_chunk = "".join(sentence_chunk).replace("  ", " ").strip()
        joined_sentence_chunk = re.sub(r'\.([A-Z])', r'. \1', joined_sentence_chunk) 
        chunk_dict["sentence_chunk"] = joined_sentence_chunk
        
        # Statistics
        chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
        chunk_dict["chunk_word_count"] = len(joined_sentence_chunk.split(" "))
        chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4 
        
        pages_and_chunks.append(chunk_dict)

print(f"Total chunks: {len(pages_and_chunks)}")

In [ ]:
df = pd.DataFrame(pages_and_chunks)
df.describe().round(2)

### 5. Filtering out Short Chunks

In [ ]:
min_token_length = 30
pages_and_chunks_over_min_token_len = df[df["chunk_token_count"] > min_token_length].to_dict(orient="records")
print(f"Filtered down to {len(pages_and_chunks_over_min_token_len)} chunks.")

### 6. Vector Embedding Generation

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load sentence-transformers model (CPU optimized)
embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2", device=device)
print(f"Loaded embedding model onto device: {device}")

In [ ]:
# Embed chunks in batches for efficiency
text_chunks = [item["sentence_chunk"] for item in pages_and_chunks_over_min_token_len]

text_chunk_embeddings = embedding_model.encode(
    text_chunks,
    batch_size=32,
    convert_to_tensor=True
)
print(f"Generated embeddings matrix of shape: {text_chunk_embeddings.shape}")

In [ ]:
# Save chunks and embeddings to file
for i, item in enumerate(pages_and_chunks_over_min_token_len):
    # Convert tensor embedding to list so it can be saved in CSV/JSON
    item["embedding"] = text_chunk_embeddings[i].cpu().tolist()

text_chunks_and_embeddings_df = pd.DataFrame(pages_and_chunks_over_min_token_len)
embeddings_df_save_path = "text_chunks_and_embeddings_df.csv"
text_chunks_and_embeddings_df.to_csv(embeddings_df_save_path, index=False)
print(f"Saved embeddings to {embeddings_df_save_path}")

### 7. Similarity Search (Retrieval)

In [ ]:
import numpy as np
import torch
from sentence_transformers import util

# Load saved embeddings and convert back to numpy arrays
loaded_df = pd.read_csv("text_chunks_and_embeddings_df.csv")
loaded_df["embedding"] = loaded_df["embedding"].apply(lambda x: np.fromstring(x.strip("[]"), sep=", "))

pages_and_chunks = loaded_df.to_dict(orient="records")
embeddings = torch.tensor(np.array(loaded_df["embedding"].tolist()), dtype=torch.float32).to(device)

In [ ]:
def retrieve_relevant_resources(query: str,
                                embeddings: torch.tensor,
                                model: SentenceTransformer = embedding_model,
                                n_resources_to_return: int = 5):
    """
    Embeds query and performs dot product similarity search against index.
    """
    query_embedding = model.encode(query, convert_to_tensor=True)
    dot_scores = util.dot_score(query_embedding, embeddings)[0]
    scores, indices = torch.topk(input=dot_scores, k=n_resources_to_return)
    return scores, indices

In [ ]:
query = "What is the classification of NBFCs?"
scores, indices = retrieve_relevant_resources(query, embeddings)

for score, index in zip(scores, indices):
    print(f"[Score: {score:.4f}] Page {pages_and_chunks[index]['page_number']}")
    print(pages_and_chunks[index]['sentence_chunk'][:300] + "...")
    print("-"*50)

### 8. Generation Step (Local and Gemini API)

#### Few-Shot Prompt Formatter

In [ ]:
def prompt_formatter(query: str, context_items: list[dict]) -> str:
    context = "- " + "\n- ".join([item["sentence_chunk"] for item in context_items])
    
    base_prompt = """Based on the following regulatory context from the Reserve Bank of India (RBI), please answer the query.
Make sure your answers are explanatory, professional, and reference specific sections or terms if present in the context.

Example 1:
Query: What is the regulatory minimum Capital Adequacy Ratio (CAR) for NBFCs?
Answer: According to RBI guidelines, non-banking financial companies (NBFCs) are required to maintain a minimum Capital to Risk-Weighted Assets Ratio (CRAR) or Capital Adequacy Ratio (CAR). For most systemically important non-deposit taking NBFCs and deposit-taking NBFCs, the regulatory minimum is set at 15%. This capital adequacy framework is designed to ensure that the NBFC retains a sufficient capital buffer against operational and credit risks, thereby preserving financial stability within the fintech and lending ecosystem.

Example 2:
Query: What is the Upper Layer classification criteria?
Answer: The Upper Layer (NBFC-UL) of non-banking financial companies comprises those entities which are specifically identified by the RBI as warranting enhanced regulatory discipline. This identification is based on a set of parameters including size, leverage, nature of operations, connectivity to the financial system, and group structure. Typically, the top 10 eligible NBFCs by asset size are automatically classified in the Upper Layer, while others are selected based on a multi-factor scoring methodology covering interconnectedness, complexity, and systemic risk.

Now use the following context items to answer the user query:
{context}

User query: {query}
Answer:"""
    return base_prompt.format(context=context, query=query)

#### Option A: Local LLM setup (Requires a GPU / runs slowly on CPU)

In [ ]:
# In the original tutorial, an instruction-tuned model like google/gemma-2b-it or gemma-7b-it is used.
# Here we outline how to load a CPU-friendly model if you choose to execute it locally:
"""
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

inputs = tokenizer(formatted_prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=256)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
"""

#### Option B: Google Gemini API (High-performance CPU Fallback)

In [ ]:
import google.generativeai as genai

# Ensure your GEMINI_API_KEY environment variable is set
api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    print("[WARNING] GEMINI_API_KEY environment variable not found. Please set it or enter it in the web interface.")
else:
    genai.configure(api_key=api_key)

def generate_answer_via_api(prompt: str) -> str:
    if not os.environ.get("GEMINI_API_KEY"):
        return "Error: Gemini API Key not set. Please set GEMINI_API_KEY in your environment."
    
    model = genai.GenerativeModel('gemini-1.5-flash')
    response = model.generate_content(prompt)
    return response.text

#### Complete RAG Ask Function

In [ ]:
def ask(query: str, use_api: bool = True) -> tuple[str, list[dict]]:
    # 1. Retrieve top context
    scores, indices = retrieve_relevant_resources(query, embeddings)
    context_items = [pages_and_chunks[idx] for idx in indices]
    
    # 2. Format prompt
    formatted_prompt = prompt_formatter(query, context_items)
    
    # 3. Generate answer
    if use_api:
        answer = generate_answer_via_api(formatted_prompt)
    else:
        answer = "[Local Generation Mode placeholder - configure local model if local execution is desired]"
        
    return answer, context_items

In [ ]:
query = "What NBFCs are included in the Middle Layer?"
if os.environ.get("GEMINI_API_KEY"):
    ans, contexts = ask(query)
    print(f"Query: {query}\n")
    print(f"Answer:\n{ans}\n")
    print(f"Retrieved from page: {contexts[0]['page_number']}")
else:
    print("GEMINI_API_KEY environment variable not set. Set the key to run this cell.")